# 02 · Features IRIS

In [ ]:
%pip install -q geopandas xlrd openpyxl scikit-learn requests pyarrow

In [ ]:
ROOT = "/lakehouse/default"
RAYON_M = 350
PERIODE_TRAIN = (2015, 2020)
PERIODE_RECENTE = (2021, 2025)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape
from shapely import wkt
import json

read = lambda name: spark.read.table(name).toPandas()

def save(df, name):
    spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    print(name, len(df))

In [ ]:
iris = read("bronze_iris_geo")
iris = gpd.GeoDataFrame(iris, geometry=iris.geometry_json.apply(lambda g: shape(json.loads(g))), crs=4326).to_crs(2154)
iris["aire_km2"] = iris.area / 1e6
centres = iris.set_geometry(iris.centroid)[["code_iris", "geometry"]]

def pop(year):
    p = read(f"bronze_pop_{year}").rename(columns=lambda c: c.replace(f"P{year % 100}_", "").lower() if c.startswith("P") else c)
    p = p.rename(columns={"IRIS": "code_iris"}).drop(columns="COM")
    p["part_0014"] = p["pop0014"] / p["pop"]
    p["part_1529"] = p["pop1529"] / p["pop"]
    p["part_60p"] = (p["pop6074"] + p["pop75p"]) / p["pop"]
    return p[["code_iris", "pop", "part_0014", "part_1529", "part_60p"]].add_suffix(f"_{year}").rename(columns={f"code_iris_{year}": "code_iris"})

df = iris.drop(columns=["geometry_json"]).merge(pop(2021), on="code_iris", how="left").merge(pop(2015), on="code_iris", how="left")
df["densite_2015"] = df.pop_2015 / df.aire_km2
df["densite_2021"] = df.pop_2021 / df.aire_km2
df["tx_annuel_15_21"] = (df.pop_2021 / df.pop_2015) ** (1 / 6) - 1

In [ ]:
log = read("bronze_sitadel").dropna(subset=["lat"])
log = log[log.ETAT_DAU != "4"]
log["annee"] = log.DATE_REELLE_AUTORISATION.str[:4].astype(int)
log = gpd.sjoin(gpd.GeoDataFrame(log, geometry=gpd.points_from_xy(log.lon, log.lat), crs=4326).to_crs(2154), iris[["code_iris", "geometry"]], predicate="within")
for year, (a, b) in [(2015, PERIODE_TRAIN), (2021, PERIODE_RECENTE)]:
    lgt = log[log.annee.between(a, b)].groupby("code_iris").NB_LGT_TOT_CREES.sum().rename(f"lgt_autorises_{a}_{b}")
    df = df.merge(lgt, on="code_iris", how="left").fillna({f"lgt_autorises_{a}_{b}": 0})
    df[f"lgt_par_hab_an_{year}"] = df[f"lgt_autorises_{a}_{b}"] / (b - a + 1) / df[f"pop_{year}"]

In [ ]:
car = read("bronze_carreaux")
car = gpd.GeoDataFrame(car, geometry=gpd.points_from_xy(car.x, car.y), crs=3035).to_crs(2154)
car = gpd.sjoin(car, iris[["code_iris", "geometry"]], predicate="within").drop(columns="index_right")
qpv = read("bronze_qpv")
qpv = gpd.GeoDataFrame(qpv, geometry=qpv.geometry_wkt.apply(wkt.loads), crs=4326).to_crs(2154)
car["qpv"] = car.index.isin(gpd.sjoin(car[["geometry"]], qpv[["geometry"]], predicate="within").index)

s = car.groupby("code_iris")[["ind", "men", "men_pauv", "ind_snv"]].sum()
socio = pd.DataFrame({"ind_carreaux": s.ind, "taux_pauvrete": s.men_pauv / s.men, "niveau_vie_moyen": s.ind_snv / s.ind, "qpv": car.groupby("code_iris").qpv.any()})
lg = read("bronze_logement_2021").rename(columns={"IRIS": "code_iris"})
socio = socio.join(lg.set_index("code_iris").eval("P21_RP_VOIT1P / P21_RP").rename("part_menages_voiture"))
df = df.merge(socio, on="code_iris", how="left")

def acces(table, lat, lon, nom):
    pts = read(table)
    pts = gpd.GeoDataFrame(pts, geometry=gpd.points_from_xy(pts[lon], pts[lat]), crs=4326).to_crs(2154)[["geometry"]]
    d = f"dist_{nom}_m"
    car[d] = gpd.sjoin_nearest(car[["geometry"]], pts, distance_col=d).groupby(level=0)[d].min()
    nc = car[car[d] > RAYON_M]
    paires = gpd.sjoin(nc[["code_iris", "geometry"]].set_geometry(nc.buffer(RAYON_M)), nc[["code_iris", "ind", "geometry"]], predicate="contains")
    gain = paires[paires.code_iris_left == paires.code_iris_right].groupby(level=0).ind.sum()
    best = nc.assign(gain=gain).sort_values("gain").groupby("code_iris").tail(1)
    pos = best.geometry.to_crs(4326)
    best = pd.DataFrame({"code_iris": best.code_iris, f"gain_ind_{nom}": best.gain, f"site_lat_{nom}": pos.y, f"site_lon_{nom}": pos.x})
    w = car.assign(c=car.ind * (car[d] <= RAYON_M), w=car.ind * car[d]).groupby("code_iris")[["ind", "c", "w"]].sum()
    out = pd.DataFrame({f"part_pop_desservie_{nom}": w.c / w.ind, f"dist_pond_{nom}_m": w.w / w.ind})
    return out.reset_index().merge(best, on="code_iris", how="left").fillna({f"gain_ind_{nom}": 0})

for table, lat, lon, nom in [("bronze_stops", "stop_lat", "stop_lon", "arret_bus"), ("bronze_citiz", "lat", "lon", "citiz"), ("bronze_velo", "lat", "lon", "velo")]:
    df = df.merge(acces(table, lat, lon, nom), on="code_iris", how="left")

df["geometry_wkt"] = df.geometry.to_crs(4326).to_wkt()
centre = df.geometry.centroid.to_crs(4326)
df["lat"], df["lon"] = centre.y, centre.x
silver = pd.DataFrame(df.drop(columns="geometry"))
save(silver, "silver_iris")
silver.groupby("ville")[["pop_2021", "ind_carreaux", "lgt_autorises_2015_2020", "lgt_autorises_2021_2025", "qpv"]].sum()